# Comunidades propagadoras: execução determinística da heurística GRASP

Este notebook executa a heurística na rede completa congelada do baseline: 20.099 vértices e 31.545 arcos. Com `p=1`, a função objetivo é o alcance externo calculado por `ExactReach`. O MIA fica para a etapa futura com `p<1`.

Execute as células em ordem no diretório do projeto, usando o ambiente `dbpedia-network`. São 30 construções e buscas locais para cada um dos dois valores de `γ`, com sementes 0 a 29. Cada execução concluída é salva em `community_results/heuristic/exact_reach/`; repetir uma célula continua de onde parou. A última célula salva até 20 soluções elite por `γ`. A comparação com Leiden será feita depois.

In [ ]:
import csv
import hashlib
import json
import os
import time
from pathlib import Path

from community_heuristic import (
    CommunitySolution, ExactReach, cpm_score, diverse_elite,
    grasp_construct, grasp_refine, internal_arc_count,
    is_cpm_feasible, is_weakly_connected, load_baseline_graph,
)

project_root = Path.cwd()
assert (project_root / 'dbpedia-network-canonical.graphml').is_file(), 'Abra o notebook no diretório do projeto.'
graph = load_baseline_graph(project_root)
start = time.monotonic()
exact = ExactReach(graph)
print(f'Rede: {len(graph):,} vértices, {graph.number_of_edges():,} arcos')
print(f'ExactReach preparado em {time.monotonic() - start:.1f} s')

## Parâmetros e identidade dos dados

Os dois valores de `γ` são o percentil 25 e a mediana da densidade interna dirigida das 96 comunidades Leiden. A lista restrita de candidatos contém os 20% melhores candidatos viáveis; `α=0` na restrição CPM. A primeira adição forma um par conectado mesmo quando o alcance externo cai; depois disso, cada adição exige ganho positivo. O identificador da configuração inclui os arquivos de entrada e o código da heurística para impedir que uma retomada misture versões diferentes.

In [ ]:
baseline_path = project_root / 'community_results/baseline/baseline_communities.csv'
with baseline_path.open(newline='', encoding='utf-8') as handle:
    baseline_rows = list(csv.DictReader(handle))
assert len(baseline_rows) == 96
densities = sorted(
    (int(row['internal_influence_edges']) + int(row['internal_doctoral_supervision_edges']))
    / (int(row['community_size']) * (int(row['community_size']) - 1))
    for row in baseline_rows
)
gammas = {'p25': densities[23], 'mediana': (densities[47] + densities[48]) / 2}
run_seeds = tuple(range(30))
restricted_fraction = 0.2
elite_limit = 20
max_jaccard = 0.5
results_dir = project_root / 'community_results/heuristic/exact_reach'

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

config = {
    'method': 'grasp_exact_reach_v1',
    'graph_sha256': sha256_file(project_root / 'dbpedia-network-canonical.graphml'),
    'partition_sha256': sha256_file(project_root / 'community_results/final_partitions/leiden_consensus_final.csv'),
    'baseline_communities_sha256': sha256_file(baseline_path),
    'heuristic_code_sha256': sha256_file(project_root / 'community_heuristic.py'),
    'vertices': len(graph), 'arcs': graph.number_of_edges(),
    'gammas': gammas, 'run_seeds': list(run_seeds),
    'restricted_fraction': restricted_fraction,
    'elite_limit': elite_limit, 'max_jaccard': max_jaccard,
    'score': 'ExactReach.external', 'probability': 1.0, 'alpha': 0,
}
config_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode('utf-8')).hexdigest()
print('γ:', gammas)
print('Identificador da configuração:', config_id[:16])

## Execução retomável

Cada arquivo `run_XX.json` guarda a semente, os vértices, o alcance, a coesão e os tempos da construção e da busca local. Se uma semente não formar comunidade com pelo menos dois vértices, seu arquivo registra essa tentativa. A gravação é atômica: uma interrupção durante uma execução deixa para repetir somente aquela semente.

In [ ]:
def save_json_atomic(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)

def check_manifest():
    path = results_dir / 'config.json'
    if path.exists():
        with path.open(encoding='utf-8') as handle:
            existing = json.load(handle)
        if existing != config:
            raise ValueError(f'Configuração existente diferente em {path}; use outro diretório de saída.')
    else:
        save_json_atomic(path, config)

def describe_solution(solution):
    nodes = solution.nodes
    size = len(nodes)
    internal_arcs = internal_arc_count(graph, nodes)
    total_reach = exact.total(nodes)
    assert size >= 2 and is_weakly_connected(graph, nodes)
    assert is_cpm_feasible(graph, nodes, solution.gamma)
    assert solution.proxy_score == exact.external(nodes)
    return {
        'nodes': sorted(nodes), 'size': size, 'internal_arcs': internal_arcs,
        'directed_density': internal_arcs / (size * (size - 1)),
        'cpm_score': cpm_score(graph, nodes, solution.gamma),
        'total_reach': total_reach, 'external_reach': total_reach - size,
        'external_reach_per_node': (total_reach - size) / size,
    }

def execute_gamma(label):
    check_manifest()
    gamma = gammas[label]
    run_dir = results_dir / label
    run_dir.mkdir(parents=True, exist_ok=True)
    for seed in run_seeds:
        path = run_dir / f'run_{seed:02d}.json'
        if path.exists():
            with path.open(encoding='utf-8') as handle:
                saved = json.load(handle)
            if (saved.get('config_id'), saved.get('gamma_label'), saved.get('run_seed')) != (config_id, label, seed):
                raise ValueError(f'Resultado incompatível em {path}')
            print(f'{label} / semente {seed:02d}: já concluída', flush=True)
            continue
        start = time.monotonic()
        constructed = grasp_construct(graph, exact.external, gamma, seed, restricted_fraction)
        construction_seconds = time.monotonic() - start
        result = {'config_id': config_id, 'gamma_label': label, 'gamma': gamma,
                  'run_seed': seed, 'construction_seconds': construction_seconds}
        if constructed is None:
            result.update(status='no_solution', refinement_seconds=0.0, solution=None)
        else:
            start = time.monotonic()
            refined = grasp_refine(graph, exact.external, constructed)
            result.update(status='ok', refinement_seconds=time.monotonic() - start,
                          constructed=describe_solution(constructed),
                          solution=describe_solution(refined))
        save_json_atomic(path, result)
        print(f"{label} / semente {seed:02d}: {result['status']} em "
              f"{construction_seconds + result['refinement_seconds']:.1f} s", flush=True)

check_manifest()
print('Saída:', results_dir)

## 30 execuções: `γ` no percentil 25

Esta célula pode demorar. Ao interromper, execute novamente as células de preparação e depois esta célula; os resultados concluídos serão reutilizados.

In [ ]:
execute_gamma('p25')

## 30 execuções: `γ` na mediana

Usa as mesmas sementes, a mesma rede e o mesmo escore da célula anterior.

In [ ]:
execute_gamma('mediana')

## Consolidar as soluções elite

Execute após concluir as duas células anteriores. Esta etapa seleciona até 20 comunidades por `γ`, com Jaccard de vértices no máximo 0,5 entre cada par, e salva um índice dos 60 resultados. Ela ainda não compara os dois valores de `γ` nem as comunidades Leiden.

In [ ]:
check_manifest()
run_index = []
for label, gamma in gammas.items():
    records = []
    for seed in run_seeds:
        path = results_dir / label / f'run_{seed:02d}.json'
        if not path.is_file():
            raise FileNotFoundError(f'Execução pendente: {path}')
        with path.open(encoding='utf-8') as handle:
            record = json.load(handle)
        if (record.get('config_id'), record.get('gamma_label'), record.get('run_seed')) != (config_id, label, seed):
            raise ValueError(f'Resultado incompatível em {path}')
        records.append(record)
        run_index.append({key: record[key] for key in (
            'gamma_label', 'gamma', 'run_seed', 'status',
            'construction_seconds', 'refinement_seconds')})
    solutions = [
        CommunitySolution(frozenset(record['solution']['nodes']),
                          record['solution']['external_reach'], gamma, record['run_seed'])
        for record in records if record['status'] == 'ok'
    ]
    elite = diverse_elite(solutions, limit=elite_limit, max_jaccard=max_jaccard)
    elite_records = [
        {'rank': rank, 'run_seed': solution.run_seed, **describe_solution(solution)}
        for rank, solution in enumerate(elite, start=1)
    ]
    save_json_atomic(results_dir / f'elite_{label}.json', {
        'config_id': config_id, 'gamma_label': label, 'gamma': gamma,
        'runs_completed': len(records), 'valid_solutions': len(solutions),
        'elite': elite_records,
    })
    print(f'{label}: {len(solutions)} soluções válidas; {len(elite)} na elite')
save_json_atomic(results_dir / 'run_index.json', {'config_id': config_id, 'runs': run_index})
print('Concluído. Resultados em:', results_dir)